In [3]:
from pathlib import Path
import pandas as pd

raw_dir = Path("C:/Users/visva/Desktop/Folder/Document/GUVI/Projects/P1/data/raw")

In [4]:
files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

In [5]:
dataset={}

In [6]:
for table_name, file_name in files.items():
    dataset[table_name]=pd.read_csv(raw_dir/file_name)

print ("Loaded Tables", list(dataset.keys()))

Loaded Tables ['customers', 'geolocation', 'order_items', 'payments', 'reviews', 'orders', 'products', 'sellers', 'category_translation']


In [7]:
summary=[]

for table_name,df in dataset.items():
    summary.append({
        "name": table_name, 
        "rows":df.shape[0], 
        "columns": df.shape[1], 
        "missing_cells":df.isnull().sum().sum(), 
        "duplicate_rows":df.duplicated().sum()
    })
summary_df=pd.DataFrame(summary)
summary_df


,name,rows,columns,missing_cells,duplicate_rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [8]:
for table_name, df in dataset.items():
    print (f"\n------{table_name.upper()}-----")

    column_report=pd.DataFrame({
        "data_type":df.dtypes.astype(str),
        "missing_values":df.isnull().sum(),
        "missing_percentage":(df.isnull().sum()/len(df)*100).round(2),
        "unique_values":df.nunique(dropna=False)
    })
    display(column_report)


------CUSTOMERS-----


,data_type,missing_values,missing_percentage,unique_values
customer_id,str,0,0.0,99441
customer_unique_id,str,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,str,0,0.0,4119
customer_state,str,0,0.0,27



------GEOLOCATION-----


,data_type,missing_values,missing_percentage,unique_values
geolocation_zip_code_prefix,int64,0,0.0,19015
geolocation_lat,float64,0,0.0,717360
geolocation_lng,float64,0,0.0,717613
geolocation_city,str,0,0.0,8011
geolocation_state,str,0,0.0,27



------ORDER_ITEMS-----


,data_type,missing_values,missing_percentage,unique_values
order_id,str,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,str,0,0.0,32951
seller_id,str,0,0.0,3095
shipping_limit_date,str,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999


... (8 outputs left out)


------SELLERS-----


,data_type,missing_values,missing_percentage,unique_values
seller_id,str,0,0.0,3095
seller_zip_code_prefix,int64,0,0.0,2246
seller_city,str,0,0.0,611
seller_state,str,0,0.0,23



------CATEGORY_TRANSLATION-----


,data_type,missing_values,missing_percentage,unique_values
product_category_name,str,0,0.0,71
product_category_name_english,str,0,0.0,71


In [9]:
table_understanding = pd.DataFrame({
    "table": [
        "customers",
        "geolocation",
        "order_items",
        "payments",
        "reviews",
        "orders",
        "products",
        "sellers",
        "category_translation"
    ],

    "one_row_represents": [
        "One customer record associated with an order",
        "One geographical coordinate for a postal-code prefix",
        "One product item within an order",
        "One payment made for an order",
        "One customer review associated with an order",
        "One customer order",
        "One product",
        "One seller",
        "One Portuguese-to-English product category translation"
    ],

    "candidate_key": [
        "customer_id",
        "No reliable natural key",
        "order_id + order_item_id",
        "order_id + payment_sequential",
        "review_id + order_id",
        "order_id",
        "product_id",
        "seller_id",
        "product_category_name"
    ]
})

table_understanding

,table,one_row_represents,candidate_key
0,customers,One customer record associated with an order,customer_id
1,geolocation,One geographical coordinate for a postal-code ...,No reliable natural key
2,order_items,One product item within an order,order_id + order_item_id
3,payments,One payment made for an order,order_id + payment_sequential
4,reviews,One customer review associated with an order,review_id + order_id
5,orders,One customer order,order_id
6,products,One product,product_id
7,sellers,One seller,seller_id
8,category_translation,One Portuguese-to-English product category tra...,product_category_name


In [10]:
key_checks = {
    "customers": ["customer_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

for table_name, key_columns in key_checks.items():
    df = dataset[table_name]

    duplicate_keys = df.duplicated(
        subset=key_columns
    ).sum()

    print(
        table_name,
        key_columns,
        "duplicate keys:",
        duplicate_keys
    )

customers ['customer_id'] duplicate keys: 0
order_items ['order_id', 'order_item_id'] duplicate keys: 0
payments ['order_id', 'payment_sequential'] duplicate keys: 0
reviews ['review_id', 'order_id'] duplicate keys: 0
orders ['order_id'] duplicate keys: 0
products ['product_id'] duplicate keys: 0
sellers ['seller_id'] duplicate keys: 0
category_translation ['product_category_name'] duplicate keys: 0


In [11]:
for table_name, df in dataset.items():
    missing=df.isnull().sum()
    missing=missing[missing>0]

    if not missing.empty:
        print(f"-----{table_name.upper()}-----")
        display(pd.DataFrame({
            "missing_value":missing,
            "missing_percentage":(missing/len(df)*100).round(2)
        }))

-----REVIEWS-----


,missing_value,missing_percentage
review_comment_title,87656,88.34
review_comment_message,58247,58.70


-----ORDERS-----


,missing_value,missing_percentage
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


-----PRODUCTS-----


,missing_value,missing_percentage
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


NameError: name 'customers' is not defined